# GECCO LBA — overnight experiment grid

> **Title (paper / talk, ALife angle):** *The price of modularity: Pareto evolution of routed information flow in human-grounded modular learners*  
> **Subtitle:** *Discrete routing as an evolvable phenotype; cyclic task structure (Near / Far / Same); error vs. pathway separation; single-module baselines*

Runs **many** NSGA-II + single-module baseline jobs (Holton `trial_df`, cyclic A/B schedule), writes **append-only CSVs** under a timestamped run folder so you can stop/resume and inspect partial results in the morning.

## Conclusions so far (from smaller Near-only / two-condition runs)

- **Trade-off:** lower validation MSE tends to align with **lower routing separation** (more shared pathway use); pushing toward **max split** (separation ~1) costs MSE.
- **Not yet solid:** **Far** vs **Near** ordering across seeds, **Same** geometry, and **single-module** stars vs Pareto need the fuller grids below (multi-seed, init scale, longer patterns).

## Before you run

1. `cd gecco && pip install -e .`
2. Start Jupyter with a kernel whose cwd is **`gecco/`** (folder with `pyproject.toml`), or uncomment the `sys.path` cell below.
3. Set **`GRID_MODE`**: `smoke` (1), `standard` (48 with default σ list), `extensive` (336), `mega` (660) — counts assume default `INIT_SCALES_*`; edit those lists in the config cell to tune. The grid cell prints the exact job count.
4. Optional: `export GECCO_TRIAL_DF=/abs/path/to/trial_df.csv`  
5. **Restart kernel** after editing GPU env logic above, then **Run All** — PyTorch caches CUDA at first import.

**GPU:** By default the notebook sets **`CUDA_VISIBLE_DEVICES=4,5,6,7`** before importing PyTorch so jobs do not use GPUs 0–3 (restart the kernel after changing this). Logical device is **`cuda:N`** with `N` from env **`GECCO_CUDA_DEVICE`** (default `0` = first in that list, usually physical GPU 4). To pin one card: `export GECCO_CUDA_VISIBLE_DEVICES=6` or set `CUDA_VISIBLE_DEVICES` before launching Jupyter. Rough cost scales as `|grid| × population × generations × train_steps`; lower `population` / `generations` / `train_steps` if a mode is too slow.

In [1]:
# Optional: if imports fail, set GECCO_INSTALL to the folder that contains pyproject.toml
# import sys
# GECCO_INSTALL = "/path/to/Structure-Function-Analysis-of-Network-Topologies/gecco"
# if GECCO_INSTALL not in sys.path:
#     sys.path.insert(0, GECCO_INSTALL)

from __future__ import annotations

import os

# --- GPU pool: physical devices 4–7 only (must run before `import torch`).
# If CUDA_VISIBLE_DEVICES is already set (e.g. by your shell), we do not override.
# Override default pool: export GECCO_CUDA_VISIBLE_DEVICES=5
# Pick logical GPU inside that pool: export GECCO_CUDA_DEVICE=0  (0 = first visible, usually phys 4)
if "CUDA_VISIBLE_DEVICES" not in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = os.environ.get("GECCO_CUDA_VISIBLE_DEVICES", "4,5,6,7")

_CUDA_IDX = int(os.environ.get("GECCO_CUDA_DEVICE", "0"))

import csv
import json
import time
import traceback
from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import product
from pathlib import Path
from typing import Any, Dict, Iterable, List, Tuple

import pandas as pd
import torch

from gecco.data.holton_trials import load_trial_df, pick_participants
from gecco.evolve.nsga2 import run_nsga2
from gecco.experiments.run_lba_figure import aggregate_cyclic_tensors, pareto_indices, _default_trial_df
from gecco.models.routing_rnn import DualRouteRNN, SingleRouteRNN, routing_separation_score
from gecco.training.episode import apply_init_scale, train_on_schedule

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration — edit `GRID_MODE` and GA budget here

In [2]:
# --- user-facing knobs ---
GRID_MODE = "extensive"  # "smoke" | "standard" | "extensive" | "mega"

# σ for evolved dual-path nets (`apply_init_scale`). Wider lists = more runs (resume-safe CSV).
INIT_SCALES_STANDARD = [0.0001, 0.001, 0.01, 0.1, 1.0, 2.0]  # used with near/far × 2 seeds × 2 ppc
INIT_SCALES_EXTENSIVE = [0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 2.0]  # log-ish + mid + high
INIT_SCALES_MEGA = [0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0]

# GA + training (align with gecco/scripts/overnight/_common.sh for production-like runs)
BASE = {
    "population": 44,
    "generations": 24,
    "train_steps": 180,
    "batch_size": 32,
    "lr": 0.02,
    "hidden": 32,
    "baseline_init_low": 0.001,
    "baseline_init_high": 2.0,
    "use_cpu": False,
}

# Smaller budget for smoke only
SMOKE_GA = {"population": 12, "generations": 4, "train_steps": 40}

TRIAL_DF = Path(os.environ.get("GECCO_TRIAL_DF", str(_default_trial_df()))).expanduser()

_cwd = Path.cwd()
if (_cwd / "pyproject.toml").is_file():
    _gecco_root = _cwd
elif (_cwd.parent / "pyproject.toml").is_file():
    _gecco_root = _cwd.parent
else:
    _gecco_root = _cwd

RUN_ROOT = Path(os.environ.get("GECCO_OVERNIGHT_RUN_DIR", "")).expanduser()
if not str(RUN_ROOT):
    RUN_ROOT = _gecco_root / "runs" / "overnight_nb"

# Resume: set to an existing folder that already contains manifest.csv (or set env GECCO_OVERNIGHT_RESUME)
_resume = os.environ.get("GECCO_OVERNIGHT_RESUME", "").strip()
RESUME_RUN_DIR = Path(_resume).expanduser() if _resume else None  # or assign a Path here

STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
if RESUME_RUN_DIR is not None:
    RUN_DIR = RESUME_RUN_DIR.resolve()
    RUN_DIR.mkdir(parents=True, exist_ok=True)
else:
    RUN_DIR = (RUN_ROOT / STAMP).resolve()
    RUN_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_CSV = RUN_DIR / "manifest.csv"
PARETO_CSV = RUN_DIR / "pareto_points.csv"
POP_SAMPLE_CSV = RUN_DIR / "population_sample.csv"  # optional lightweight sample

if BASE["use_cpu"] or not torch.cuda.is_available():
    device = torch.device("cpu")
else:
    n = torch.cuda.device_count()
    idx = min(max(_CUDA_IDX, 0), n - 1) if n else 0
    device = torch.device(f"cuda:{idx}")

print("trial_df:", TRIAL_DF, "exists:", TRIAL_DF.is_file())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "(unset)"))
print("device:", device, end="")
if device.type == "cuda":
    print(f" | {torch.cuda.get_device_name(device)} | torch sees {torch.cuda.device_count()} GPU(s)")
else:
    print()
print("RUN_DIR:", RUN_DIR.resolve())

trial_df: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/transfer-interference/data/participants/trial_df.csv exists: True
CUDA_VISIBLE_DEVICES: 4,5,6,7
device: cuda:0 | NVIDIA RTX 6000 Ada Generation | torch sees 4 GPU(s)
RUN_DIR: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/gecco/notebooks/20260404_024022_UTC


## Build experiment list

In [3]:
@dataclass(frozen=True)
class ExperimentSpec:
    condition: str
    seed: int
    init_scale: float
    cyclic_pattern: str
    participants_per_condition: int

    def run_id(self) -> str:
        pat = self.cyclic_pattern.replace("/", "")
        return (
            f"c{self.condition}_s{self.seed}_i{self.init_scale}_p{pat}_ppc{self.participants_per_condition}"
        )


def build_experiments(mode: str) -> List[ExperimentSpec]:
    if mode == "smoke":
        return [
            ExperimentSpec("near", 0, 0.1, "ABABAB", 2),
        ]
    elif mode == "standard":
        conds = ["near", "far"]
        seeds = [0, 1]
        inits = list(INIT_SCALES_STANDARD)
        pats = ["ABABAB"]
        ppc = [2, 3]
    elif mode == "extensive":
        conds = ["near", "far", "same"]
        seeds = [0, 1, 2, 3]
        inits = list(INIT_SCALES_EXTENSIVE)
        pats = ["ABABAB", "ABABABAB"]
        ppc = [2, 3]
    elif mode == "mega":
        conds = ["near", "far", "same"]
        seeds = [0, 1, 2, 3, 4]
        inits = list(INIT_SCALES_MEGA)
        pats = ["ABABAB", "ABABABAB"]
        ppc = [2, 3]
    else:
        raise ValueError(f"Unknown GRID_MODE: {mode!r}")

    out: List[ExperimentSpec] = []
    for c, s, i, pat, p in product(conds, seeds, inits, pats, ppc):
        out.append(ExperimentSpec(str(c), int(s), float(i), str(pat), int(p)))
    return out


experiments = build_experiments(GRID_MODE)
print(f"GRID_MODE={GRID_MODE!r} -> {len(experiments)} NSGA jobs (one condition each)")
print(f"  init_scale values: {sorted({e.init_scale for e in experiments})}")

# Persist grid for reproducibility (do not clobber on resume)
grid_path = RUN_DIR / "grid.jsonl"
if not grid_path.is_file():
    with grid_path.open("w", encoding="utf-8") as f:
        for ex in experiments:
            f.write(json.dumps(ex.__dict__, sort_keys=True) + "\n")
    print("wrote", grid_path)
else:
    print("keep existing", grid_path)

GRID_MODE='extensive' -> 96 NSGA jobs (one condition each)
wrote /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/gecco/notebooks/20260404_024022_UTC/grid.jsonl


## Core run + CSV append (resume-safe)

In [4]:
def _ensure_csv(path: Path, fieldnames: List[str]) -> None:
    if path.is_file():
        return
    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        w.writeheader()


def load_completed_run_ids(manifest: Path) -> set:
    if not manifest.is_file():
        return set()
    df = pd.read_csv(manifest)
    ok = df.loc[df.get("status", "") == "ok", "run_id"]
    return set(ok.astype(str).tolist())


def make_evaluate(tensors: Dict[str, torch.Tensor], cfg: dict):
    def evaluate(genome: List[int]) -> Tuple[float, float]:
        m = DualRouteRNN(genome, hidden_per_module=cfg["hidden"])
        apply_init_scale(m, cfg["init_scale"])
        _, val = train_on_schedule(
            m,
            tensors,
            steps=cfg["train_steps"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            device=device,
            seed=cfg["seed"] + (hash(tuple(genome)) % 997),
        )
        sep = routing_separation_score(genome)
        return val, -sep

    return evaluate


def run_baselines(tensors: Dict[str, torch.Tensor], cfg: dict) -> List[Tuple[str, float]]:
    rows = []
    for sigma, tag in [(cfg["baseline_init_low"], "low_sigma"), (cfg["baseline_init_high"], "high_sigma")]:
        sm = SingleRouteRNN(hidden_per_module=cfg["hidden"])
        apply_init_scale(sm, sigma)
        _, val = train_on_schedule(
            sm,
            tensors,
            steps=cfg["train_steps"] * 2,
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            device=device,
            seed=cfg["seed"],
        )
        rows.append((tag, float(val)))
    return rows


MANIFEST_FIELDS = [
    "run_id",
    "status",
    "error",
    "duration_s",
    "utc_finished",
    "condition",
    "seed",
    "init_scale",
    "cyclic_pattern",
    "participants_per_condition",
    "population",
    "generations",
    "train_steps",
    "participants_json",
    "pareto_n",
    "pop_n",
    "best_mse_pareto",
    "worst_mse_pareto",
    "max_routing_sep_pareto",
    "baseline_low_sigma_mse",
    "baseline_high_sigma_mse",
]

PARETO_FIELDS = [
    "run_id",
    "condition",
    "seed",
    "init_scale",
    "cyclic_pattern",
    "participants_per_condition",
    "mse",
    "f2_neg_sep",
    "routing_separation",
    "genome",
]

SAMPLE_FIELDS = [
    "run_id",
    "condition",
    "mse",
    "routing_separation",
    "genome",
]

_ensure_csv(MANIFEST_CSV, MANIFEST_FIELDS)
_ensure_csv(PARETO_CSV, PARETO_FIELDS)
_ensure_csv(POP_SAMPLE_CSV, SAMPLE_FIELDS)


def append_rows(path: Path, fieldnames: List[str], rows: Iterable[Dict[str, Any]]) -> None:
    with path.open("a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames)
        for r in rows:
            w.writerow(r)
        f.flush()


def run_one(ex: ExperimentSpec, cfg: dict) -> Tuple[Dict[str, Any], List[Dict[str, Any]], List[Dict[str, Any]]]:
    pids = pick_participants(
        df,
        conditions=[ex.condition],
        per_condition=ex.participants_per_condition,
        seed=ex.seed,
    )
    if len(pids) < ex.participants_per_condition:
        raise RuntimeError(f"only {len(pids)} participants for {ex.condition}: {pids}")

    tensors = aggregate_cyclic_tensors(
        df.loc[df["participant"].isin(pids)],
        pids,
        max_trials_per_segment=24,
        pattern=ex.cyclic_pattern,
        seed=ex.seed,
    )

    evaluate = make_evaluate(tensors, cfg)
    pop = run_nsga2(
        evaluate,
        genome_length=12,
        population_size=cfg["population"],
        generations=cfg["generations"],
        seed=cfg["seed"] + hash(ex.condition) % 10000,
    )
    front = pareto_indices(pop)
    baselines = run_baselines(tensors, cfg)

    rid = ex.run_id()
    mses = [ind.f1 for ind in front]
    seps = [routing_separation_score(ind.genome) for ind in front]
    bl = dict(baselines)

    manifest_row = {
        "run_id": rid,
        "status": "ok",
        "error": "",
        "duration_s": "",  # filled in by caller after timing
        "utc_finished": datetime.now(timezone.utc).isoformat(),
        "condition": ex.condition,
        "seed": ex.seed,
        "init_scale": ex.init_scale,
        "cyclic_pattern": ex.cyclic_pattern,
        "participants_per_condition": ex.participants_per_condition,
        "population": cfg["population"],
        "generations": cfg["generations"],
        "train_steps": cfg["train_steps"],
        "participants_json": json.dumps(pids),
        "pareto_n": len(front),
        "pop_n": len(pop),
        "best_mse_pareto": min(mses) if mses else "",
        "worst_mse_pareto": max(mses) if mses else "",
        "max_routing_sep_pareto": max(seps) if seps else "",
        "baseline_low_sigma_mse": bl.get("low_sigma", ""),
        "baseline_high_sigma_mse": bl.get("high_sigma", ""),
    }

    pareto_rows = []
    for ind in front:
        g = ind.genome
        pareto_rows.append(
            {
                "run_id": rid,
                "condition": ex.condition,
                "seed": ex.seed,
                "init_scale": ex.init_scale,
                "cyclic_pattern": ex.cyclic_pattern,
                "participants_per_condition": ex.participants_per_condition,
                "mse": ind.f1,
                "f2_neg_sep": ind.f2,
                "routing_separation": routing_separation_score(g),
                "genome": "".join(str(x) for x in g),
            }
        )

    # up to 80 individuals sampled for size control
    sample_rows = []
    for ind in pop[:80]:
        g = ind.genome
        sample_rows.append(
            {
                "run_id": rid,
                "condition": ex.condition,
                "mse": ind.f1,
                "routing_separation": routing_separation_score(g),
                "genome": "".join(str(x) for x in g),
            }
        )

    return manifest_row, pareto_rows, sample_rows

## Load data and execute grid

**Resume:** In the config cell set `RESUME_RUN_DIR = Path(".../runs/overnight_nb/<STAMP>_UTC")` **or** export `GECCO_OVERNIGHT_RESUME=/abs/path/to/that/folder`. Re-run cells in order; `grid.jsonl` is not overwritten if it already exists.

In [ ]:
if not TRIAL_DF.is_file():
    raise FileNotFoundError(
        f"Missing {TRIAL_DF} — set GECCO_TRIAL_DF or use monorepo transfer-interference trial_df.csv"
    )

df = load_trial_df(str(TRIAL_DF))

cfg = {**BASE, "init_scale": 0.1}  # placeholder; overwritten per experiment
if GRID_MODE == "smoke":
    cfg = {**cfg, **SMOKE_GA}

done = load_completed_run_ids(MANIFEST_CSV)
print("already completed (ok):", len(done))

errors = []
for ex in tqdm(experiments, desc="overnight grid"):
    rid = ex.run_id()
    if rid in done:
        continue

    row_cfg = {
        **cfg,
        "seed": ex.seed,
        "init_scale": ex.init_scale,
    }
    t0 = time.perf_counter()
    try:
        manifest_row, pareto_rows, sample_rows = run_one(ex, row_cfg)
        dt = time.perf_counter() - t0
        manifest_row["duration_s"] = f"{dt:.3f}"
        manifest_row["utc_finished"] = datetime.now(timezone.utc).isoformat()
        append_rows(MANIFEST_CSV, MANIFEST_FIELDS, [manifest_row])
        append_rows(PARETO_CSV, PARETO_FIELDS, pareto_rows)
        append_rows(POP_SAMPLE_CSV, SAMPLE_FIELDS, sample_rows)
        print(f"OK {rid} ({dt:.1f}s)")
    except Exception as e:
        dt = time.perf_counter() - t0
        err = traceback.format_exc()
        append_rows(
            MANIFEST_CSV,
            MANIFEST_FIELDS,
            [
                {
                    "run_id": rid,
                    "status": "error",
                    "error": repr(e),
                    "duration_s": f"{dt:.3f}",
                    "utc_finished": datetime.now(timezone.utc).isoformat(),
                    "condition": ex.condition,
                    "seed": ex.seed,
                    "init_scale": ex.init_scale,
                    "cyclic_pattern": ex.cyclic_pattern,
                    "participants_per_condition": ex.participants_per_condition,
                    "population": cfg["population"],
                    "generations": cfg["generations"],
                    "train_steps": cfg["train_steps"],
                    "participants_json": "",
                    "pareto_n": "",
                    "pop_n": "",
                    "best_mse_pareto": "",
                    "worst_mse_pareto": "",
                    "max_routing_sep_pareto": "",
                    "baseline_low_sigma_mse": "",
                    "baseline_high_sigma_mse": "",
                }
            ],
        )
        with (RUN_DIR / "tracebacks.txt").open("a", encoding="utf-8") as f:
            f.write(f"\n===== {rid} =====\n{err}\n")
        errors.append((rid, repr(e)))
        print(f"ERR {rid}: {e}")

print("done. errors:", len(errors))
if errors:
    print(errors[:5], "...")

already completed (ok): 0


overnight grid:   0%|          | 0/96 [00:00<?, ?it/s]overnight grid:   1%|          | 1/96 [31:33<49:58:34, 1893.84s/it]

OK cnear_s0_i0.001_pABABAB_ppc2 (1893.8s)


overnight grid:   2%|▏         | 2/96 [1:03:24<49:42:20, 1903.62s/it]

OK cnear_s0_i0.001_pABABAB_ppc3 (1910.5s)


...

OK cnear_s2_i2.0_pABABAB_ppc2 (1453.6s)


overnight grid:  23%|██▎       | 22/96 [9:29:57<30:32:09, 1485.54s/it]

OK cnear_s2_i2.0_pABABAB_ppc3 (1502.4s)


overnight grid:  24%|██▍       | 23/96 [9:51:40<29:00:43, 1430.74s/it]

OK cnear_s2_i2.0_pABABABAB_ppc2 (1302.9s)


## Quick readout (after partial or full run)

In [ ]:
m = pd.read_csv(MANIFEST_CSV)
display(m.tail(8))
if PARETO_CSV.is_file() and PARETO_CSV.stat().st_size > 0:
    p = pd.read_csv(PARETO_CSV)
    print("pareto rows:", len(p))
    display(p.groupby("condition")["mse"].agg(["count", "min", "median"]).reset_index())